# Chapter 6: Matrix Decompositions

Factoring a matrix into simpler parts reveals structure and enables efficient algorithms.

| Decomposition | Formula | Best For |
|---|---|---|
| LU | $PA = LU$ | Solving systems, reusing factorisation |
| QR | $A = QR$ | Least squares (stable), eigenvalue algorithms |
| Cholesky | $A = LL^T$ | PD systems, Gaussian sampling |
| Eigendecomposition | $A = PDP^{-1}$ | Understanding structure, matrix powers |
| SVD | $A = U\Sigma V^T$ | **Everything else** — any matrix |

## Learning Objectives
- Perform LU, QR, Cholesky decompositions and verify properties
- Understand SVD as the universal decomposition for any matrix
- Apply low-rank approximation (Eckart-Young theorem)
- Compress an image using SVD


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


## LU Decomposition

$$PA = LU$$

- $P$: permutation matrix (row swaps for numerical stability)
- $L$: lower triangular with 1s on diagonal
- $U$: upper triangular

This is organised Gaussian elimination. Advantage: **factorise $A$ once, solve for many $b$** efficiently.

Solving $Ax = b$ via LU:
1. $Ly = Pb$ (forward substitution — $O(n^2)$)
2. $Ux = y$ (back substitution — $O(n^2)$)


In [ ]:
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)
b = np.array([1., 1., 1.])

P_mat, L, U = sla.lu(A)
print("P =\n", P_mat)
print("L =\n", L)
print("U =\n", U)
print("P@A == L@U?", np.allclose(P_mat @ A, L @ U))

# Solve using forward then back substitution
y  = sla.solve_triangular(L, P_mat @ b, lower=True)
x_lu = sla.solve_triangular(U, y, lower=False)
x_direct = np.linalg.solve(A, b)
print("\nLU solution:", x_lu)
print("Direct solve:", x_direct)
print("Match?", np.allclose(x_lu, x_direct))


## QR Decomposition

$$A = QR$$

- $Q$: orthogonal matrix ($Q^TQ = I$)
- $R$: upper triangular

QR is the formal version of Gram-Schmidt. Uses:
- **Least squares** (more stable than normal equations)
- Core of the **QR algorithm** for computing eigenvalues

Solving $\min\|Ax-b\|^2$ via QR: since $Q$ is orthogonal, $\min\|QRx-b\|^2 = \min\|Rx-Q^Tb\|^2$.


In [ ]:
A = np.random.randn(5, 3)
b = np.random.randn(5)

Q, R = np.linalg.qr(A)
print("Q^T Q = I?", np.allclose(Q.T @ Q, np.eye(3)))
print("R is upper triangular?\n", np.round(R, 4))

# Least squares via QR (stable)
x_qr = sla.solve_triangular(R, Q.T @ b, lower=False)
x_lstsq, *_ = np.linalg.lstsq(A, b, rcond=None)
print("\nQR solution:", x_qr)
print("lstsq:      ", x_lstsq)
print("Match?", np.allclose(x_qr, x_lstsq, atol=1e-10))


## Cholesky Decomposition

For symmetric **positive definite** $A$:

$$A = LL^T$$

where $L$ is lower triangular with positive diagonal entries.

- **2× faster** than LU for PD matrices
- Applications: Gaussian processes, Kalman filter, sampling multivariate Gaussians
- To sample $x \sim \mathcal{N}(0, \Sigma)$: compute $L = \text{chol}(\Sigma)$, then $x = Lz$ where $z \sim \mathcal{N}(0,I)$


In [ ]:
np.random.seed(42)

# Create a PD covariance matrix
Sigma = np.array([[4, 2], [2, 3]], dtype=float)
L_chol = np.linalg.cholesky(Sigma)
print("L =\n", L_chol)
print("L @ L^T == Sigma?", np.allclose(L_chol @ L_chol.T, Sigma))

# Sample from N(0, Sigma) using Cholesky
n_samples = 1000
z = np.random.randn(2, n_samples)
x_samples = L_chol @ z

print(f"\nEmpirical covariance (should ≈ Sigma):\n{np.cov(x_samples)}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*x_samples, alpha=0.2, s=8)
ax.set(aspect='equal', title=r'Samples from $\mathcal{N}(0, \Sigma)$ via Cholesky')
plt.tight_layout(); plt.show()


## Singular Value Decomposition (SVD)

The **most powerful** decomposition — works for **any** $m \times n$ matrix:

$$A = U \Sigma V^T$$

| Factor | Shape | Properties |
|---|---|---|
| $U$ | $m \times m$ | Left singular vectors (orthogonal) |
| $\Sigma$ | $m \times n$ | Diagonal, singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ |
| $V^T$ | $n \times n$ | Right singular vectors (orthogonal) |

**Relationship to eigenvalues:**
$$\sigma_i = \sqrt{\lambda_i(A^T A)}, \quad U = \text{eigenvectors of } AA^T, \quad V = \text{eigenvectors of } A^TA$$


In [ ]:
A = np.array([[1, 2, 0],
              [3, 1, 2],
              [2, 0, 1]], dtype=float)

U, s, Vt = np.linalg.svd(A)
print("U =\n", U)
print("Singular values:", s)
print("V^T =\n", Vt)

# Reconstruct
Sigma_full = np.zeros_like(A)
np.fill_diagonal(Sigma_full, s)
A_reconstructed = U @ Sigma_full @ Vt
print("\nReconstruction error:", np.linalg.norm(A - A_reconstructed))

# Verify sigma_i^2 = eigenvalues of A^T A
eigs_ATA = np.linalg.eigvalsh(A.T @ A)[::-1]  # descending
print("\nσ²:", np.round(s**2, 6))
print("λ(A^TA):", np.round(eigs_ATA, 6))
print("Match?", np.allclose(s**2, eigs_ATA, atol=1e-8))


## Geometry of SVD

Every linear transformation $A$ decomposes into three steps:

$$A = \underbrace{U}_{\text{rotate/reflect}} \cdot \underbrace{\Sigma}_{\text{scale axes}} \cdot \underbrace{V^T}_{\text{rotate/reflect}}$$

Visualise by transforming the **unit circle** through each stage.


In [ ]:
A_2d = np.array([[3, 1], [1, 2]], dtype=float)
U2, s2, Vt2 = np.linalg.svd(A_2d)

# Unit circle
theta = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(theta), np.sin(theta)])

stage1 = Vt2 @ circle                       # after V^T: rotate
stage2 = np.diag(s2) @ stage1               # after Sigma: scale
stage3 = U2 @ stage2                         # after U: final rotation = A@circle

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, pts, title, col in zip(axes,
    [circle, stage1, stage2, stage3],
    ['Unit Circle', r'After $V^T$ (rotate)', r'After $\Sigma$ (scale)', r'After $U$ (= $Ax$)'],
    ['steelblue', 'green', 'orange', 'red']):
    ax.plot(*pts, color=col, lw=2)
    ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
    ax.set(xlim=(-4,4), ylim=(-4,4), aspect='equal', title=title)
    ax.grid(True)
plt.suptitle('Geometry of SVD: Unit Circle Through Each Stage', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## Low-Rank Approximation

**Eckart-Young Theorem:** The best rank-$k$ approximation to $A$ (in Frobenius norm) is:

$$A_k = U_k \Sigma_k V_k^T = \sum_{i=1}^k \sigma_i u_i v_i^T$$

Compression ratio: original $mn$ values → $k(m + n + 1)$ values.

The singular values tell us **how much information** is in each rank-1 component.


In [ ]:
A_lr = np.random.randn(20, 15)
U_lr, s_lr, Vt_lr = np.linalg.svd(A_lr, full_matrices=False)
r = len(s_lr)

errors = []
explained = []
for k in range(1, r+1):
    A_k = U_lr[:,:k] @ np.diag(s_lr[:k]) @ Vt_lr[:k,:]
    errors.append(np.linalg.norm(A_lr - A_k, 'fro'))
    explained.append(np.sum(s_lr[:k]**2) / np.sum(s_lr**2) * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, r+1), errors, 'o-', color='tomato')
ax1.set(title='Frobenius Error vs Rank', xlabel='rank k', ylabel=r'$\|A - A_k\|_F$')

ax2.plot(range(1, r+1), explained, 's-', color='steelblue')
ax2.axhline(90, ls='--', color='gray', label='90% threshold')
ax2.set(title='Cumulative Explained Variance', xlabel='rank k', ylabel='% variance')
ax2.legend()
plt.tight_layout(); plt.show()


## Image Compression with SVD

A grayscale image is just a matrix of pixel values. SVD decomposes it into rank-1 components.
Keeping only the top-$k$ components reconstructs a compressed approximation.

Most visual information lives in the **first few singular values**.


In [ ]:
from matplotlib.image import imread as mpl_imread

# Generate a synthetic 128x128 test image (Gaussian blobs)
np.random.seed(0)
img_size = 128
img = np.zeros((img_size, img_size))
for _ in range(6):
    cx, cy = np.random.randint(20, 108, 2)
    sigma  = np.random.randint(8, 20)
    xx, yy = np.meshgrid(np.arange(img_size), np.arange(img_size))
    img   += np.exp(-((xx-cx)**2 + (yy-cy)**2) / (2*sigma**2))
img /= img.max()

U_img, s_img, Vt_img = np.linalg.svd(img, full_matrices=False)

ranks = [5, 15, 40, len(s_img)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, k in zip(axes, ranks):
    A_k = U_img[:,:k] @ np.diag(s_img[:k]) @ Vt_img[:k,:]
    mn = img_size; orig_vals = mn * mn
    comp_vals = k * (mn + mn + 1)
    ratio = orig_vals / comp_vals
    ax.imshow(A_k, cmap='gray', vmin=0, vmax=1)
    ax.set(title=f'rank={k}\ncompression {ratio:.1f}×' if k < len(s_img)
                 else f'rank={k} (full)\noriginal', xticks=[], yticks=[])
plt.suptitle('SVD Image Compression', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Summary

| Decomposition | When to Use |
|---|---|
| LU | Solve $Ax=b$ for multiple $b$, general square $A$ |
| QR | Least squares (stable), eigenvalue algorithms |
| Cholesky | PD matrices: fast solve, multivariate Gaussian sampling |
| Eigendecomposition | Understand structure, matrix powers, Markov chains |
| SVD | Any matrix: compression, PCA, pseudoinverse, LSA |

Rule of thumb: **if in doubt, use SVD**.

**Next:** Chapter 7 — Applications in ML & AI: PCA, regression, recommendations, NLP, and neural networks.
